# **Agent's tools**

## **Imports and environment configs**

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from openai import OpenAI

MODEL = "gpt-5.4-mini"

openai_client = OpenAI()

In [3]:
import json

from bibliometric_rag_assistant.ingestion.pubmed import search_pubmed
from bibliometric_rag_assistant.search_strategy import PUBMED_QUERY

## **Simple Agent - answers without tools**

In [4]:
history = [
    {"role": "user", "content": "What is an agent, properly?"}
]


response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=history
)

print(response.output_text)

An **agent**, properly, is something that **acts on behalf of something else** or **has the capacity to act intentionally in an environment**.

Depending on context, the word has a few related meanings:

1. **Everyday / legal sense**  
   An agent is a person or entity authorized to represent another.  
   - Example: a travel agent, a real-estate agent, an attorney-in-fact.

2. **Philosophical sense**  
   An agent is a being that can **choose, intend, and do things for reasons**.  
   - It is not merely a passive object; it initiates actions.
   - This is often tied to concepts like responsibility, free will, and intention.

3. **AI / computer science sense**  
   An agent is a system that **perceives its environment and acts to achieve goals**.  
   - Example: a thermostat, a robot vacuum, or a language-model-based assistant.
   - In this sense, an agent typically has:
     - **perception/input**
     - **a policy or decision process**
     - **action/output**
     - often some **obj

- `Response`: class;
- `response`: class instance;

When we call the instance `response`, the output is a text representation of the class from which this instance belongs to:

In [5]:
response

Response(id='resp_08b18917ea0f8939006ad279f8474487d2bd584bf1c690280e', created_at=1792178680.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseOutputMessage(id='msg_08b18917ea0f8939006ad279f9389087d2bd4909cfb202a16a', content=[ResponseOutputText(annotations=[], text='An **agent**, properly, is something that **acts on behalf of something else** or **has the capacity to act intentionally in an environment**.\n\nDepending on context, the word has a few related meanings:\n\n1. **Everyday / legal sense**  \n   An agent is a person or entity authorized to represent another.  \n   - Example: a travel agent, a real-estate agent, an attorney-in-fact.\n\n2. **Philosophical sense**  \n   An agent is a being that can **choose, intend, and do things for reasons**.  \n   - It is not merely a passive object; it initiates actions.\n   - This is often tied to concepts like responsibility, free will, and intention

In [7]:
history

[{'role': 'user', 'content': 'What is an agent, properly?'}]

In [8]:
history += response.output

history.append({"role": "user", "content": "What specifically is an AI agent?"})

second_response = openai_client.responses.create(
    model=MODEL,
    input=history,
)

print(second_response.output_text)

An **AI agent** is a system that **perceives information, decides what to do, and takes actions to achieve a goal**.

More specifically, it usually has these parts:

- **Inputs / observations**: it receives data from the world
- **Decision-making**: it chooses actions based on that data
- **Actions**: it does something that changes the world or its own state
- **Objective**: it is trying to optimize or accomplish something

A simple formal way to say it:

> An AI agent maps observations to actions in order to maximize expected performance on some task.

Examples:
- A **robot vacuum** senses walls and dirt, then decides where to move.
- A **chess engine** observes the board, evaluates positions, and chooses a move.
- A **customer-support chatbot** reads a message, decides how to respond, and sends text back.

### What makes something “agentic”?
An AI system is more agent-like when it can:
- act **autonomously**
- make **multi-step decisions**
- pursue **goals over time**
- use tools or 

In [11]:
history += second_response.output

history.append({"role": "user", "content": "According with the last answers, will you become an agent when i provide tools and call you in a loop?"})

third_response = openai_client.responses.create(
    model=MODEL,
    input=history,
)

print(third_response.output_text)

Yes — **if** you give me tools and run me in a loop, I can function **as an agent-like system**.

But there’s an important distinction:

### What I am by default
I’m a **model**: I generate outputs in response to inputs.

### What makes me agent-like
If you add:
- **tools** I can call,
- a **loop** that lets me observe results and continue,
- and some **goal** or task objective,

then I can participate in a process that has the usual structure of an AI agent:
1. observe
2. decide
3. act
4. observe again
5. repeat

### What that means in practice
In that setup, I’m not becoming a completely different kind of thing, but I am being used as the **decision-making core** of an agent.

So the agent is really the **whole system**:
- the model
- the tools
- the loop/controller
- the environment
- the goal

### Important caveat
I still don’t have independent agency in the human sense. I:
- don’t initiate activity on my own unless invoked
- don’t have my own persistent desires or intentions
- don

## **Defining a tool**

- **General json schema for callable functions as tools**

In [12]:
search_pubmed_tool = {
    "type": "function",
    "name": "search_pubmed",
    "description": (
        "Search PubMed and return the total number of matching records "
        "and a limited list of PMIDs. Use this for PubMed literature searches."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": (
                    "PubMed query using its Boolean operators and field tags, "
                    "such as [Title/Abstract] and [MeSH Terms]."
                ),
            },
            "limit": {
                "type": "integer",
                "minimum": 1,
                "maximum": 20,
                "description": (
                    "Maximum number of PMIDs to return. "
                    "Use 20 when the user does not specify a limit."
                ),
            },
        },
        "required": ["query", "limit"],
        "additionalProperties": False,
    },
    "strict": True,
}


## **Sending the question with the tool**

Mesmo com a ferramenta, vai responder "na lata". Não precisa de decidir se chama o não a ferramenta, já que a pergunta não tem a menor relação com o que a ferramenta pode fazer. 

In [13]:
history = [
    {"role": "user", "content": "What is an agent, properly?"}
]


response = openai_client.responses.create(
    model=MODEL,
    input=history,
    tools=[search_pubmed_tool]  # SCHEMA/JSON definition
)

for item in response.output:
    print(item.type, item)

message ResponseOutputMessage(id='msg_06b0e4382cb9fdae006ad2828c1bec87d2984fc109c1bf31e7', content=[ResponseOutputText(annotations=[], text='An **agent**, properly, is **something that acts on its own behalf in the world**.\n\nMore precisely, in philosophy and AI, an agent is typically understood as an entity that:\n\n1. **Perceives** its environment somehow  \n2. **Forms some internal state or goals**\n3. **Selects actions** in light of those goals\n4. **Acts** to affect the world\n5. Does so with at least some degree of **autonomy**\n\n### A compact definition\nAn agent is a system that **chooses actions to achieve objectives under conditions of uncertainty**.\n\n### What makes something an agent?\nCommon features include:\n\n- **Autonomy**: it is not merely pushed around by inputs; it initiates action.\n- **Goal-directedness**: its behavior is organized around outcomes.\n- **Responsiveness**: it adapts to changes in its environment.\n- **Persistence over time**: it can maintain a tr

o `.type` diz que espécie de saída está sendo examinada!

In [9]:
response.output_text

'“Agent” can mean different things depending on the field, but in the most general sense:\n\n**An agent is something that acts on behalf of a goal, and whose actions are responsive to its environment.**\n\nA good, broad definition usually includes these pieces:\n\n1. **It perceives or observes** something about the world.\n2. **It chooses actions** based on what it perceives.\n3. **Its actions affect outcomes** in the world.\n4. **It is organized around some objective, preference, or function**—even if only implicitly.\n\n### In philosophy and everyday language\nAn agent is a being that can **intend, decide, and act**.  \nA person is a paradigm agent because they can deliberate and choose among alternatives.\n\n### In economics and social science\nAn agent is an entity that **makes decisions under constraints**—for example, a consumer, firm, or government. The focus is on **behavior and incentives**.\n\n### In AI and computer science\nAn agent is usually a system that:\n- receives inpu

Agora a pergunta terá relação com a funcionalidade da ferramenta, então na primeira chamada o modelo aprende o `args` que deve ser usado numa eventual chamada da ferramenta, mas a chamada em si só ocorre na segunda interação com a api.

- **Primeira chamada: o modelo decide se usa ou não uma tool**

In [22]:
instructions = f"You are helping analyze a PubMed literature review. The current search strategy's query is: {PUBMED_QUERY}"


history = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": "Which pmids returns from the query in effect for this pipeline?"}
]


response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=history,
    tools=[search_pubmed_tool]  # SCHEMA/JSON definition
)

for item in response.output:
    print(item.type, item)

function_call ResponseFunctionToolCall(arguments='{"query":"( \\"Temporomandibular Joint\\"[Mesh] OR \\"temporomandibular joint\\"[Title/Abstract] ) AND ( \\"Joint Prosthesis\\"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( \\"2000/01/01\\"[Date - Publication] : \\"2025/12/31\\"[Date - Publication] )","limit":20}', call_id='call_haBdNq5cYNYnOhvYqFZZcPkR', name='search_pubmed', type='function_call', id='fc_08bb2ebe5c70485a006ad2834df54087d289f56b35d74aaf45', async_=None, caller=None, namespace=None, status='completed')


- **Resultado da primeira chamada: `arguments`**

Não há uma regra fixa, o modelo pode retornar em `arguments` a pergunta do usuário reformulada ou simplesmente copiar um query literal contido em `instructions` - nesse caso temos o `PUBMED_QUERY`.

In [38]:
call = response.output[0]
args = json.loads(call.arguments) # Json -> Python object (a dict here!)

results = search_pubmed(**args) # HTTP request happens here, the unpacking just deliver the arguments
result_json = json.dumps(results, indent=2) # Python object (a dict here!) -> Json

result_json

'{\n  "header": {\n    "type": "esearch",\n    "version": "0.3"\n  },\n  "esearchresult": {\n    "count": "100",\n    "retmax": "20",\n    "retstart": "0",\n    "idlist": [\n      "41328123",\n      "41076314",\n      "40725551",\n      "40393895",\n      "40335417",\n      "40153056",\n      "40090550",\n      "39756382",\n      "39490003",\n      "39487073",\n      "39344460",\n      "39344453",\n      "39299812",\n      "39256068",\n      "39232864",\n      "38743036",\n      "38685147",\n      "38071499",\n      "37847424",\n      "37246472"\n    ],\n    "translationset": [],\n    "querytranslation": "(\\"Temporomandibular Joint\\"[MeSH Terms] OR \\"Temporomandibular Joint\\"[Title/Abstract]) AND (\\"Joint Prosthesis\\"[MeSH Terms] OR \\"prosthe*\\"[Title/Abstract] OR \\"replacement\\"[Title/Abstract] OR \\"alloplastic\\"[Title/Abstract]) AND (\\"infection*\\"[Title/Abstract] OR \\"antibiotic*\\"[Title/Abstract] OR \\"prophylaxis\\"[Title/Abstract]) AND 2000/01/01:2025/12/31[Date -

O objeto `function_call` inteiro (o `ResponseFunctionToolCall`) é o que entra em history, não só o texto ou os argumentos - é justamente esse objeto completo (com `.call_id`, `.name`, `.arguments`) que a API depois precisa ver de volta no histórico pra saber a quê o `function_call_output` que você manda em seguida está respondendo.

In [ ]:
history += response.output

history.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})


response = openai_client.responses.create(
    model=MODEL,
    input=history,
    tools=[search_pubmed_tool],
)

response.output_text

'The query returns **100 PubMed records** total.\n\nThe first 20 PMIDs returned are:\n\n1. 41328123  \n2. 41076314  \n3. 40725551  \n4. 40393895  \n5. 40335417  \n6. 40153056  \n7. 40090550  \n8. 39756382  \n9. 39490003  \n10. 39487073  \n11. 39344460  \n12. 39344453  \n13. 39299812  \n14. 39256068  \n15. 39232864  \n16. 38743036  \n17. 38685147  \n18. 38071499  \n19. 37847424  \n20. 37246472\n\nIf you want, I can also help retrieve the next batches of PMIDs beyond the first 20.'

## **Agentic loop**

In [ ]:
def make_call(call):
    # JSON string -> python dict
    args = json.loads(call.arguments)

    if call.name == "search_pubmed":
        result = search_pubmed(**args)

    # Turn results into JSON
    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json
    },

In [14]:
MAX_TOOL_CALLS = 2

instructions = f"You are helping analyze a PubMed literature review. The current search strategy's query is: {PUBMED_QUERY}"


def run_agent(question: str) -> dict:
    history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    call_log = []
    stop_reason = None
    tool_call_count = 0
    final_answer = None
    it = 0

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=MODEL,
            input=history,
            tools=[search_pubmed_tool],
        )

        history.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                if tool_call_count >= MAX_TOOL_CALLS:
                    stop_reason = 'max_tool_calls'
                    break

                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                tool_call_count += 1

                call_log.append({
                    'tool': item.name,
                    'arguments': json.loads(item.arguments),
                    'result_summary': call_output['output'][:200],
                })

                history.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                final_answer = item.content[0].text
                print('ASSISTANT:')
                print(final_answer)

        if stop_reason == 'max_tool_calls':
            print(f'Parando: limite de {MAX_TOOL_CALLS} chamadas de tool atingido.')
            break

        it += 1
        if not has_function_calls:
            stop_reason = 'end_turn'
            break

    return {
        'answer': final_answer,
        'call_log': call_log,
        'stop_reason': stop_reason,
    }

## **Testando com três perguntas conhecidas**

A terceira pergunta não tem relação nenhuma com PubMed — serve pra confirmar que o agente também sabe **não** chamar a tool quando ela não é necessária.

In [15]:
known_questions = [
    "How many PMIDs does the query in effect for this pipeline return?",
    "What is the first PMID returned by our current search?",
    "What is the capital of France?",
]

for question in known_questions:
    print("=" * 60)
    print("QUESTION:", question)
    result = run_agent(question)
    print("STOP REASON:", result["stop_reason"])
    print("CALL LOG:", result["call_log"])
    print("ANSWER:", result["answer"])
    print()

QUESTION: How many PMIDs does the query in effect for this pipeline return?
iteration #0...
function_call: search_pubmed {"query":"( \"Temporomandibular Joint\"[Mesh] OR \"temporomandibular joint\"[Title/Abstract] ) AND ( \"Joint Prosthesis\"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( \"2000/01/01\"[Date - Publication] : \"2025/12/31\"[Date - Publication] )","limit":20}
iteration #1...
ASSISTANT:
The query returns **100 PMIDs** in total.
STOP REASON: end_turn
CALL LOG: [{'tool': 'search_pubmed', 'arguments': {'query': '( "Temporomandibular Joint"[Mesh] OR "temporomandibular joint"[Title/Abstract] ) AND ( "Joint Prosthesis"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( "2000/01/0